## Notebook37

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes, element_text
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
storm = pl.read_csv(ub + "data/storms.csv")
storm_type = pl.read_csv(ub + "data/storm_codes.csv")

### Questions

`storm` is NOAA's best-track history of North Atlantic tropical cyclones, one
row per observation logged roughly every six hours since 1950 (more often
around landfall, as you will see). `year`, `month`, `day`, and `hour` split
the observation time into separate numeric components, the same way the
book recommends recording dates. `name` is the storm's common name, `letter`
its single-letter season code, and `doy` the day of the year. `lon` and `lat`
give the storm center's position, `status` a two-letter code for its phase
(tropical depression, tropical storm, hurricane, and so on), `category` its
Saffir-Simpson category (-1 for anything below hurricane strength, 0 for a
tropical storm, 1 through 5 for a hurricane), and `wind` its maximum
sustained wind in knots. `storm_type` maps each `status` code to a readable
name. One thing worth knowing before question 1: Atlantic storm names are
recycled on a six-year rotation, so `name` alone does not pick out a single
storm. Print results unless a question says otherwise.

1. Filter `storm` to rows where `name` is `"Katrina"`, without filtering on
`year` yet. Add a row number with `.with_row_index("row_number")` and plot
`wind` against `row_number` with `geom_line()`. Then select the distinct
`year` values in this filtered table. What do they tell you about the plot
you just drew?

**Answer**:


2. Filter to the real 2005 storm (`name == "Katrina"` and `year == 2005`) and
save it as `kat`. Build a fractional year column, extended to account for
`hour` the way the chapter describes, and plot it against `wind`.

3. Add a real `date` column to `kat` with `pl.date(c.year, c.month, c.day)`
and overwrite `kat` with it. Plot `date` against `wind`.

4. Redraw the plot from question 3 with `scale_x_date()`, breaking the axis
every day and labeling it `"%b %d"`, angling the labels 45 degrees so they
do not collide. Then sort `kat` by `wind`, descending, and print the date,
wind, category, and status of the top 5 rows.

Katrina peaked at 150 knots, a category 5, on August 28. Notice that the
staircase shape in the plot above comes from days that hold more than one
observation stacked at the same x position, a hint that `date` is coarser
than the data actually is.

5. Using the full `storm` table (not just `kat`), build a `date` column,
extract the weekday with `.dt.weekday()`, and group by weekday to compute
the average `wind` and a count. Sort by weekday.

The average wind barely moves across the seven days, staying between 51 and
53 knots. That is exactly what should happen: unlike Wikipedia page views,
which rise and fall with when people happen to be browsing, a storm's
intensity has no reason to know what day of the week it is. A flat result
here is confirmation the method works, not a finding on its own.

6. Filter `kat` to the observations with `date` after August 26, 2005, using
`pl.date()` to build the comparison value, and print `date`, `hour`, `wind`,
and `category`.

7. Build a `moment` column on `kat` with `pl.datetime(c.year, c.month,
c.day, c.hour)` and overwrite `kat` with it. Plot `moment` against `wind`
with `scale_x_datetime()`, same breaks and labels as question 4. Then use
`.filter()` with `.is_in()` to find the rows whose `hour` is not one of the
four routine values, 0, 6, 12, or 18.

The staircase from question 4 is gone, because `moment` carries the hour
that `date` threw away. And the six-hour rhythm is not perfectly steady:
one off-cycle fix shows up on August 25, and two more cluster on August 29,
right when Katrina was making landfall. That is exactly when forecasters
most need an observation between the routine ones.

8. Sort `kat` by `wind`, descending, and print `moment`, `wind`, and
`category` for the top 3. Then filter `kat` to `moment` strictly after the
top row's timestamp and print how many rows remain.

The peak reading itself, at 2005-08-28 18:00, does not survive a strict
`>` filter against its own timestamp. It is the same boundary the chapter's
example ran into with New Year's Day, just with hurricanes standing in for
Wikipedia edits.

9. NOAA reports these timestamps in UTC. Attach that zone to `moment` with
`.dt.replace_time_zone("UTC")`, then convert to `"America/Chicago"` (the
zone Katrina's landfall coast sits in) with `.dt.convert_time_zone()`. Find
the local Central time of the peak-wind row.

10. Now make the mirror-image mistake the chapter warns about: instead of
converting `utc` to Central time, call `.dt.replace_time_zone("America/Chicago")`
on it directly. Compare the result to the correctly converted `central`
column for the peak row, and explain why they disagree.

**Answer**:


11. Join `kat` to `storm_type` on `status` and group by `status_name` to
count how many of Katrina's 34 observations fall in each phase, sorted
descending.

Nineteen of the thirty-four observations, just over half, were logged
while Katrina was already a hurricane, not just approaching one.

12. Across the full `storm` table, count the distinct storm `name`s per
`year`, bucket the years into decades, and compute the average storms
tracked per decade. Plot the result with `geom_col()`.

The average climbs from under 9 storms a year in the 1950s to over 15 by
the 2010s, with 2020 alone averaging 21. Before reading this as storms
getting more frequent, recall the same observation-window caution the
chapter raised for Wikipedia's revision counts: satellite coverage of the
Atlantic did not begin until the mid-1960s, and tracking methods kept
improving for decades after that. Some of this rise is real. Some of it is
that the 1950s could only record what a ship or a plane happened to sail
through.